In [1]:
import sys, os
sys.path.insert(0, "/home/ubuntu/projects/Megatron-LM")
from gpt_builders import gpt_builder

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/ubuntu/projects/llm-dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch.distributed as dist

os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12356'
dist.init_process_group(backend='nccl', rank=0, world_size=1)

In [3]:
# Usage: %run [file_name] [args]
%env CUDA_DEVICE_MAX_CONNECTIONS=1
%run /home/ubuntu/projects/Megatron-LM/pretrain_gpt.py --yaml_cfg "/home/ubuntu/projects/llm-models/llm-models/models/bidirectional/gpt_config.yaml"

env: CUDA_DEVICE_MAX_CONNECTIONS=1
warning using experimental yaml arguments feature, argparse arguments will be ignored
warning using experimental yaml arguments feature, argparse arguments will be ignored
warning using experimental yaml arguments feature, argparse arguments will be ignored
using world size: 1, data-parallel size: 1, context-parallel size: 1, tensor-model-parallel size: 1, pipeline-model-parallel size: 1
accumulate and all-reduce gradients in fp32 for bfloat16 data type.
using torch.bfloat16 for parameters ...
------------------------ arguments ------------------------
  accumulate_allreduce_grads_in_fp32 .............. True
  adam_beta1 ...................................... 0.9
  adam_beta2 ...................................... 0.95
  adam_eps ........................................ 1e-08
  add_position_embedding .......................... False
  adlr_autoresume ................................. False
  adlr_autoresume_interval ........................ 1000
  ali

/home/ubuntu/projects/Megatron-LM/megatron/training/utils.py:410: UserWarning: The MSC feature is disabled.
  warnings.warn(message)


AttributeError: 'types.SimpleNamespace' object has no attribute 'tiktoken_num_special_tokens'

In [6]:
os.environ.get('CUDA_DEVICE_MAX_CONNECTIONS') != 1

True

In [3]:
torch.cuda.device_count()

1

In [4]:
import torch.distributed as dist

os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12356'
dist.init_process_group(backend='nccl', rank=0, world_size=1)

In [ ]:
from functools import partial
from megatron.training.yaml_arguments import load_yaml, core_transformer_config_from_yaml
from megatron.training import get_args
from types import SimpleNamespace

from megatron.training.initialize import initialize_megatron

# Load YAML and build transformer config once
args_from_yaml = load_yaml("/home/ubuntu/projects/llm-models/llm-models/models/bidirectional/gpt_config.yaml")
# args_dict = vars(args_from_yaml).copy()

transformer_config = core_transformer_config_from_yaml(args_from_yaml, "language_model")

transfomer_key = 'language_model'
# 1. Convert the base YAML object to a dictionary
args_dict = vars(args_from_yaml).copy()

# 2. Safely merge the nested dictionaries
# Using .update() ensures that if a key exists in both, the nested one takes precedence
transfomer_key = 'language_model'
if hasattr(args_from_yaml, transfomer_key):
    args_dict.update(vars(getattr(args_from_yaml, transfomer_key)))

if hasattr(args_from_yaml, 'model_parallel'):
    args_dict.update(vars(args_from_yaml.model_parallel))

# 3. Create the namespace from the merged dictionary
args = SimpleNamespace(**args_dict)

initialize_megatron(
    extra_args_provider=None,
    args_defaults=vars(args),
    ignore_unknown_args=True # Useful if your YAML has extra fields
)

# Provide gpt_builder with that config (config=... skips internal config loading)
def model_provider_with_yaml_config(pre_process=True, post_process=True, vp_stage=None):
    # You must still have get_args() returning the full args (e.g. from same YAML)
    # args = get_args()

        # Initalize and get arguments, timers, and Tensorboard writer.
    extra_args_provider, args_defaults, get_embedding_ranks, get_position_embedding_ranks = None, None, None, None
    store = None
    
    return gpt_builder(args, pre_process, post_process, vp_stage, config=transformer_config)

model = model_provider_with_yaml_config()
# Then in pretrain:
# pretrain(..., partial(model_provider, model_provider_with_yaml_config), ...)